[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/fingerprint-similarity-guide/blob/main/notebooks/08_size_and_folding.ipynb)

# 08 · Fingerprint Size, Folding & Counts

*Fingerprint & Similarity Guide · Hamza Hentabli, Ph.D. · ~40 min*

Should you fold your Morgan fingerprint to 1,024 bits or 2,048? Use binary bits or counts? These feel like footnotes but they change results. We measure the cost of collisions directly, then let the benchmarks from Notebooks 04 and 07 settle the 1024-vs-2048 and count-vs-binary questions with real numbers.

**Learning objectives.** By the end of this notebook you will be able to:

- measure how bit collisions grow as you shrink the fingerprint
- compare 1,024- vs 2,048-bit ECFP on the virtual-screening benchmark
- compare count vs binary fingerprints on the k-NN QSAR benchmark
- choose a sensible default size and representation

In [1]:
# Setup: on Google Colab this installs dependencies and clones the repo so the
# cached data and the shared `fsg` helper module are available. Running locally,
# install requirements.txt once and this cell just locates the files.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
OFFLINE = os.environ.get("FSG_OFFLINE", "0") == "1"   # CI sets this; no notebook needs the network
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rdkit", "scikit-learn", "scipy", "pandas", "matplotlib"], check=True)
    if not os.path.isdir("fingerprint-similarity-guide"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/fingerprint-similarity-guide.git"], check=True)
    ROOT = "fingerprint-similarity-guide"
else:
    ROOT = ".." if os.path.isdir("../data") else "."
DATA_DIR = os.path.join(ROOT, "data")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import fsg   # shared fingerprint factory + metrics + evaluation

print("Colab:", IN_COLAB, "| offline:", OFFLINE, "| data:", DATA_DIR)
print("Fingerprints available:", ", ".join(fsg.FP_NAMES))

Colab: False | offline: True | data: ../data
Fingerprints available: ECFP4-1024, ECFP4-2048, ECFP6-2048, ECFP4-counts, FCFP4-2048, RDKit, AtomPair, Torsion, MACCS, Avalon


In [2]:
import json, os
import pandas as pd
DOCS_DIR = os.path.join(ROOT, "docs")
RESULTS = json.load(open(os.path.join(DOCS_DIR, "benchmark_results.json")))
print("Loaded committed benchmark results from docs/benchmark_results.json")

Loaded committed benchmark results from docs/benchmark_results.json


## 1. Collisions vs size

Folding maps many substructure environments onto a fixed number of bits; shorter fingerprints pack more environments per bit. We count collisions on a drug-sized molecule across sizes.

In [3]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator as rfg
mol = Chem.MolFromSmiles("CS(=O)(=O)CCNCc1oc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)cc1")  # lapatinib
import pandas as pd
rows = []
for size in [256, 512, 1024, 2048, 4096]:
    gen = rfg.GetMorganGenerator(radius=2, fpSize=size)
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    gen.GetFingerprint(mol, additionalOutput=ao)
    bi = ao.GetBitInfoMap()
    envs = sum(len(v) for v in bi.values()); bits = len(bi)
    collided = sum(1 for v in bi.values() if len(v) > 1)
    rows.append((size, envs, bits, collided, round(100*collided/bits, 1)))
pd.DataFrame(rows, columns=["fpSize","environments","bits_set","colliding_bits","%_colliding"])

,fpSize,environments,bits_set,colliding_bits,%_colliding
0,256,115,65,16,24.6
1,512,115,72,12,16.7
2,1024,115,73,12,16.4
3,2048,115,73,12,16.4
4,4096,115,74,11,14.9


Shrinking from 2,048 to 256 bits roughly multiplies the collision rate — distinct chemistry gets merged onto shared bits, blurring similarity.

## 2. Does 2,048 beat 1,024 for screening?

From the virtual-screening benchmark (Notebook 04).

In [4]:
b1 = RESULTS["benchmark1_virtual_screening"]["scores"]
import pandas as pd
rows = [(k, b1[f"ECFP4-{k}|tanimoto"]["BEDROC"], b1[f"ECFP4-{k}|tanimoto"]["EF1%"]) for k in ["1024","2048"]]
pd.DataFrame(rows, columns=["ECFP4 size","BEDROC","EF1%"])

,ECFP4 size,BEDROC,EF1%
0,1024,0.829,6.200
1,2048,0.849,6.208


2,048 bits give a small but real edge over 1,024 for screening — worth it, since the cost is trivial. Below ~1,024 bits the collision penalty starts to bite.

## 3. Count vs binary for QSAR

From the k-NN QSAR benchmark (Notebook 07): ECFP4 as counts vs binary.

In [5]:
b4 = RESULTS["benchmark4_knn_qsar_esol"]["scores"]
rows = [("ECFP4-counts", b4["ECFP4-counts|tanimoto"]["R2"], b4["ECFP4-counts|tanimoto"]["RMSE"]),
        ("ECFP4-2048 (binary)", b4["ECFP4-2048|tanimoto"]["R2"], b4["ECFP4-2048|tanimoto"]["RMSE"])]
pd.DataFrame(rows, columns=["representation","R2","RMSE"])

,representation,R2,RMSE
0,ECFP4-counts,0.772,1.000
1,ECFP4-2048 (binary),0.652,1.237


Counts win the regression — remembering *how many* times a feature occurs helps predict a count-sensitive property like solubility. For pure ranking/screening the gain is smaller.

**Defaults:** 2,048-bit Morgan radius 2 is a solid general choice; switch to **counts** for QSAR regression, and only drop to 1,024 bits when memory or speed forces it.

## Summary

- Collisions rise sharply below ~1,024 bits; 2,048 is a safe default.
- 2,048 > 1,024 for screening by a small, free margin.
- **Counts beat binary** for neighbour-based QSAR regression.


### ✏️ Exercise 1

Measure the collision rate of caffeine at 128 vs 2,048 bits.

<details>
<summary><b>Show solution</b></summary>

```python
for size in [128, 2048]:
    gen = rfg.GetMorganGenerator(radius=2, fpSize=size)
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    gen.GetFingerprint(Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C"), additionalOutput=ao)
    bi = ao.GetBitInfoMap()
    print(size, sum(1 for v in bi.values() if len(v) > 1), "colliding bits")
```

</details>

### ✏️ Exercise 2

From the QSAR grid, is the count advantage bigger under Tanimoto or Dice?

<details>
<summary><b>Show solution</b></summary>

```python
for m in ["tanimoto","dice"]:
    print(m, round(b4[f"ECFP4-counts|{m}"]["R2"] - b4[f"ECFP4-2048|{m}"]["R2"], 3))
```

</details>